# ⚖️ NyayAI — InLegalBERT Fine-tuning on Google Colab

This notebook fine-tunes **law-ai/InLegalBERT** as a token classifier for Indian legal document error detection.

## Labels
| Label | Meaning |
|-------|---------|
| O | Not an error |
| B-SPELL / I-SPELL | Spelling mistake |
| B-GRAM / I-GRAM | Grammar error |
| B-SEM / I-SEM | Semantic error (wrong section, clause, legal reference) |

## Steps
1. Mount Drive, install deps
2. Upload `train.jsonl`, `val.jsonl` (from your local `data/training/`)
3. Run training in chunks of 150k samples
4. Save final model to Google Drive
5. Download locally and replace `models/nyayai-error-detector/`

In [ ]:
# ── CELL 1: Mount Google Drive (for saving model) ─────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_SAVE_PATH = '/content/drive/MyDrive/NyayAI/nyayai-error-detector'
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
print(f'Model will be saved to: {DRIVE_SAVE_PATH}')

In [ ]:
# ── CELL 2: Install dependencies ──────────────────────────────────────────────
!pip install transformers==4.44.2 datasets seqeval accelerate -q
print('Dependencies installed ✓')

In [ ]:
# ── CELL 3: Verify GPU ────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → GPU')

In [ ]:
# ── CELL 4: Upload dataset files ──────────────────────────────────────────────
# Upload train.jsonl and val.jsonl from your local NyayAI/data/training/ folder
# These are small (4.1MB and 535KB respectively)

from google.colab import files
print('Select train.jsonl and val.jsonl from data/training/ on your machine')
uploaded = files.upload()  # Will open a file picker
print('Uploaded files:', list(uploaded.keys()))

In [ ]:
# ── CELL 5: Config ────────────────────────────────────────────────────────────
import json, random
from pathlib import Path

BASE_MODEL    = 'law-ai/InLegalBERT'
OUTPUT_DIR    = '/content/nyayai-error-detector'
LABEL_LIST    = ['O', 'B-SPELL', 'I-SPELL', 'B-GRAM', 'I-GRAM', 'B-SEM', 'I-SEM']
LABEL2ID      = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL      = {i: l for i, l in enumerate(LABEL_LIST)}

# Colab T4 has 16GB VRAM — use larger batch than local 4050 (6GB)
BATCH_SIZE    = 16       # Colab T4 can handle this
GRAD_ACCUM    = 2        # Effective batch = 32
MAX_LEN       = 256
EPOCHS        = 5        # More epochs = better semantic learning
LEARNING_RATE = 2e-5
CHUNK_SIZE    = 150_000  # Process dataset in chunks of 150k samples

TRAIN_FILE    = 'train.jsonl'
VAL_FILE      = 'val.jsonl'

print('Config ready ✓')
print(f'  Base model:  {BASE_MODEL}')
print(f'  Batch size:  {BATCH_SIZE} (effective {BATCH_SIZE * GRAD_ACCUM})')
print(f'  Chunk size:  {CHUNK_SIZE:,} samples')
print(f'  Epochs:      {EPOCHS}')

In [ ]:
# ── CELL 6: Count total training samples ─────────────────────────────────────
total_train = sum(1 for _ in open(TRAIN_FILE))
total_val   = sum(1 for _ in open(VAL_FILE))
n_chunks    = (total_train + CHUNK_SIZE - 1) // CHUNK_SIZE

print(f'Total training samples: {total_train:,}')
print(f'Total validation samples: {total_val:,}')
print(f'Will process in {n_chunks} chunk(s) of {CHUNK_SIZE:,}')

In [ ]:
# ── CELL 7: Load tokenizer ────────────────────────────────────────────────────
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print(f'Tokenizer loaded: {BASE_MODEL} ✓')

In [ ]:
# ── CELL 8: Dataset utilities ─────────────────────────────────────────────────
import torch
from torch.utils.data import Dataset

class NyayAIDataset(Dataset):
    """
    Token-level NER dataset for legal error detection.
    Each example: {tokens: [str, ...], labels: [str, ...]}
    """
    def __init__(self, examples, tokenizer, max_len=256):
        self.examples  = examples
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex     = self.examples[idx]
        tokens = ex.get('tokens', ex.get('words', []))
        labels = ex.get('labels', [])

        enc = self.tokenizer(
            tokens,
            is_split_into_words=True,
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )

        word_ids   = enc.word_ids(0)
        label_ids  = []
        seen_words = set()

        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)  # ignore [CLS], [SEP], [PAD]
            elif wid in seen_words:
                label_ids.append(-100)  # only label first subword
            else:
                seen_words.add(wid)
                raw_label = labels[wid] if wid < len(labels) else 'O'
                label_ids.append(LABEL2ID.get(raw_label, 0))

        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(label_ids, dtype=torch.long),
        }


def load_chunk(path: str, start: int, size: int):
    """Load a chunk of JSONL examples from `start` to `start+size`."""
    examples = []
    with open(path) as f:
        for i, line in enumerate(f):
            if i < start:
                continue
            if i >= start + size:
                break
            line = line.strip()
            if line:
                examples.append(json.loads(line))
    return examples


def load_all(path: str):
    """Load full JSONL file (for small val set)."""
    examples = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                examples.append(json.loads(line))
    return examples

print('Dataset utilities ready ✓')

In [ ]:
# ── CELL 9: Metrics ───────────────────────────────────────────────────────────
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report

def compute_metrics(p):
    preds, labels_out = p
    preds = np.argmax(preds, axis=2)

    true_preds  = []
    true_labels = []

    for pred_seq, label_seq in zip(preds, labels_out):
        p_tags = []
        l_tags = []
        for p_val, l_val in zip(pred_seq, label_seq):
            if l_val != -100:
                p_tags.append(ID2LABEL[p_val])
                l_tags.append(ID2LABEL[l_val])
        true_preds.append(p_tags)
        true_labels.append(l_tags)

    return {
        'f1':        f1_score(true_labels, true_preds),
        'precision': precision_score(true_labels, true_preds),
        'recall':    recall_score(true_labels, true_preds),
    }

print('Metrics ready ✓')

In [ ]:
# ── CELL 10: Initialize model ─────────────────────────────────────────────────
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model loaded: {total_params:.1f}M parameters on {device} ✓')

In [ ]:
# ── CELL 11: MAIN TRAINING LOOP (chunked) ────────────────────────────────────
#
# Strategy: Load data in CHUNK_SIZE (150k) chunks.
# Each chunk = one TrainingArguments run (continuing from previous checkpoint).
# This avoids Colab RAM limits while training on the full dataset.
#
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

# Load validation set once (small: ~535KB)
val_examples = load_all(VAL_FILE)
val_dataset  = NyayAIDataset(val_examples, tokenizer, MAX_LEN)
print(f'Validation set: {len(val_dataset):,} examples')

data_collator = DataCollatorForTokenClassification(tokenizer)

chunk_start = 0
chunk_idx   = 0

while chunk_start < total_train:
    chunk_idx   += 1
    is_last      = (chunk_start + CHUNK_SIZE >= total_train)
    actual_size  = min(CHUNK_SIZE, total_train - chunk_start)

    print(f'\n{'='*60}')
    print(f'CHUNK {chunk_idx}/{n_chunks} — samples {chunk_start:,} to {chunk_start + actual_size:,}')
    print(f'{'='*60}')

    train_examples = load_chunk(TRAIN_FILE, chunk_start, actual_size)
    train_dataset  = NyayAIDataset(train_examples, tokenizer, MAX_LEN)

    steps_per_epoch = max(1, len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM))

    args = TrainingArguments(
        output_dir                  = f'{OUTPUT_DIR}-chunk{chunk_idx}',
        num_train_epochs             = EPOCHS,
        per_device_train_batch_size  = BATCH_SIZE,
        per_device_eval_batch_size   = BATCH_SIZE * 2,
        gradient_accumulation_steps  = GRAD_ACCUM,
        learning_rate                = LEARNING_RATE,
        weight_decay                 = 0.01,
        warmup_ratio                 = 0.1,
        lr_scheduler_type            = 'linear',
        evaluation_strategy          = 'epoch',
        save_strategy                = 'epoch',
        load_best_model_at_end       = True,
        metric_for_best_model        = 'f1',
        greater_is_better            = True,
        save_total_limit             = 1,
        fp16                         = torch.cuda.is_available(),
        logging_steps                = max(10, steps_per_epoch // 4),
        report_to                    = 'none',
        dataloader_num_workers       = 2,
        seed                         = 42,
    )

    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = train_dataset,
        eval_dataset    = val_dataset,
        tokenizer       = tokenizer,
        data_collator   = data_collator,
        compute_metrics = compute_metrics,
    )

    trainer.train()

    # Free chunk from RAM before next iteration
    import gc
    del train_examples, train_dataset
    gc.collect()

    chunk_start += CHUNK_SIZE

print('\n🎉 Training complete on all chunks!')

In [ ]:
# ── CELL 12: Evaluate final model ────────────────────────────────────────────
results = trainer.evaluate()
print('\n📊 Final Evaluation:')
for k, v in results.items():
    print(f'   {k}: {v:.4f}' if isinstance(v, float) else f'   {k}: {v}')

In [ ]:
# ── CELL 13: Save final model to /content and Google Drive ───────────────────
import shutil, json as _json

# Save to /content first
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save training results
with open(f'{OUTPUT_DIR}/training_results.json', 'w') as f:
    _json.dump({
        'eval_f1':        results.get('eval_f1', 0),
        'eval_precision': results.get('eval_precision', 0),
        'eval_recall':    results.get('eval_recall', 0),
        'eval_loss':      results.get('eval_loss', 0),
        'epochs':         EPOCHS,
        'total_samples':  total_train,
    }, f, indent=2)

# Also save trainer_state.json for the metrics dashboard
last_chunk_dir = f'{OUTPUT_DIR}-chunk{chunk_idx}'
trainer_state  = next(Path(last_chunk_dir).rglob('trainer_state.json'), None)
if trainer_state:
    shutil.copy(str(trainer_state), f'{OUTPUT_DIR}/trainer_state.json')

# Copy to Google Drive
if os.path.exists(DRIVE_SAVE_PATH):
    shutil.rmtree(DRIVE_SAVE_PATH)
shutil.copytree(OUTPUT_DIR, DRIVE_SAVE_PATH)

print(f'\n✅ Model saved to:')
print(f'   Colab:  {OUTPUT_DIR}')
print(f'   Drive:  {DRIVE_SAVE_PATH}')
print(f'\nFiles saved:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}') // 1024
    print(f'   {f}  ({size} KB)')

In [ ]:
# ── CELL 14: Create ZIP and download ─────────────────────────────────────────
# This creates a zip of the model for easy download to your local machine

!zip -r /content/nyayai-error-detector.zip {OUTPUT_DIR}

from google.colab import files
files.download('/content/nyayai-error-detector.zip')

print('\n📦 Model ZIP downloaded!')
print('Next step: unzip and replace local models/nyayai-error-detector/')

## 🔄 How to use the trained model locally

After downloading `nyayai-error-detector.zip`:

```bash
# 1. Backup old model
mv ~/NyayAI/models/nyayai-error-detector ~/NyayAI/models/nyayai-error-detector-old

# 2. Extract new model
unzip nyayai-error-detector.zip -d ~/NyayAI/models/

# 3. Restart backend (model loads on first request)
cd ~/NyayAI
source /home/apurva/NYAYAI/nyayai/bin/activate
uvicorn src.api.main:app --host 0.0.0.0 --port 8000 --reload
```

## 📊 Expected Colab T4 Training Time

| Samples | Epochs | T4 Time |
|---------|--------|----------|
| 150,000 | 5 | ~45 min |
| 300,000 | 5 | ~90 min |
| 500,000 | 5 | ~2.5 hr |

> **Colab Pro tip:** If session disconnects, re-run from Cell 10 onwards. The model variable in RAM is lost but Drive checkpoint is safe.